# Smile / Lip ExtractorExtracts just the mouth/smile region from a face photo using a face-parsingsegmentation model loaded from Google Drive.

### 1. Mount Google Drive and load the model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install -q torch transformers pillow matplotlib opencv-python

In [ ]:
!pip install protobuf==5.29.5
!pip install tensorflow==2.19.0
!pip install mediapipe==0.10.21

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation

# Path to your face-parsing model folder in Google Drive
MODEL_PATH = "/content/drive/MyDrive/face-parsing"

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = SegformerImageProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForSemanticSegmentation.from_pretrained(MODEL_PATH).to(device)
model.eval()

print("Model loaded successfully!")

### 2. Upload a photo

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

image = Image.open(image_path).convert("RGB")
image

### 3. Run face parsing

In [ ]:
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

upsampled_logits = torch.nn.functional.interpolate(
    outputs.logits,
    size=image.size[::-1],
    mode="bilinear",
    align_corners=False,
)

labels = upsampled_logits.argmax(dim=1)[0].cpu().numpy()

### 4. Extract the smile (mouth) regionCombines the mouth, upper lip, and lower lip classes, then crops theimage tightly around that region.

In [ ]:
# Label ids from the face-parsing model: 10=mouth, 11=upper_lip, 12=lower_lip
SMILE_LABELS = [10, 11, 12]

img = np.array(image)
smile_mask = np.isin(labels, SMILE_LABELS)

ys, xs = np.where(smile_mask)
if len(xs) == 0:
    raise RuntimeError("No mouth/lip region detected in this image.")

x1, x2 = xs.min(), xs.max()
y1, y2 = ys.min(), ys.max()

smile_crop = img[y1:y2 + 1, x1:x2 + 1]

plt.figure(figsize=(5, 3))
plt.imshow(smile_crop)
plt.axis("off")
plt.title("Smile")
plt.show()

In [ ]:
# Label ids from the face-parsing model: 10=mouth, 11=upper_lip, 12=lower_lip
SMILE_LABELS = [10,]

img = np.array(image)
smile_mask = np.isin(labels, SMILE_LABELS)

ys, xs = np.where(smile_mask)
if len(xs) == 0:
    raise RuntimeError("No mouth/lip region detected in this image.")

x1, x2 = xs.min(), xs.max()
y1, y2 = ys.min(), ys.max()

smile_crop = img[y1:y2 + 1, x1:x2 + 1]

plt.figure(figsize=(5, 3))
plt.imshow(smile_crop)
plt.axis("off")
plt.title("Smile")
plt.show()

In [ ]:
# Label ids from the face-parsing model: 10=mouth, 11=upper_lip, 12=lower_lip
SMILE_LABELS = [ 11, 12]

img = np.array(image)
smile_mask = np.isin(labels, SMILE_LABELS)

ys, xs = np.where(smile_mask)
if len(xs) == 0:
    raise RuntimeError("No mouth/lip region detected in this image.")

x1, x2 = xs.min(), xs.max()
y1, y2 = ys.min(), ys.max()

smile_crop = img[y1:y2 + 1, x1:x2 + 1]

plt.figure(figsize=(5, 3))
plt.imshow(smile_crop)
plt.axis("off")
plt.title("Smile")
plt.show()

In [ ]:
# Label ids from the face-parsing model: 10=mouth, 11=upper_lip, 12=lower_lip
SMILE_LABELS = [10]

img = np.array(image)
smile_mask = np.isin(labels, SMILE_LABELS)

ys, xs = np.where(smile_mask)
if len(xs) == 0:
    raise RuntimeError("No mouth/lip region detected in this image.")

x1, x2 = xs.min(), xs.max()
y1, y2 = ys.min(), ys.max()

# Extract bounding box
smile_bbox = img[y1:y2 + 1, x1:x2 + 1]
smile_mask_cropped = smile_mask[y1:y2 + 1, x1:x2 + 1]

print(f"Smile region: x=[{x1}, {x2}], y=[{y1}, {y2}]")
print(f"Cropped size: {smile_bbox.shape}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Bounding box crop (rectangle)
axes[0].imshow(smile_bbox)
axes[0].set_title("Bounding Box Crop", fontsize=12)
axes[0].axis("off")

# 2. Segmentation mask
axes[1].imshow(smile_mask_cropped, cmap="gray")
axes[1].set_title("Segmentation Mask", fontsize=12)
axes[1].axis("off")

# 3. Extracted smile part (masked extraction)
smile_extracted = np.zeros_like(smile_bbox)
smile_extracted[smile_mask_cropped] = smile_bbox[smile_mask_cropped]
axes[2].imshow(smile_extracted)
axes[2].set_title("Extracted Smile Part", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans

# Extract mouth region pixels (RGB) - only from the mask
mouth_pixels = smile_bbox[smile_mask_cropped]

if len(mouth_pixels) == 0:
    raise RuntimeError("No mouth pixels found.")

print(f"Total mouth pixels: {len(mouth_pixels)}")

# Apply K-means clustering to find dominant colors
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans.fit(mouth_pixels)

# Get cluster centers (dominant colors)
dominant_colors = kmeans.cluster_centers_.astype(int)

# Count pixels in each cluster
labels_km = kmeans.labels_
unique, counts = np.unique(labels_km, return_counts=True)

# Sort by brightness (whiteness) - teeth are typically bright
brightness = np.sum(dominant_colors, axis=1)  # R + G + B
sorted_indices = np.argsort(brightness)[::-1]  # descending order

print("\n=== Dominant Colors in Mouth Region ===")
print(f"{'Rank':<6} {'R':<6} {'G':<6} {'B':<6} {'Brightness':<12} {'Pixels':<8}")
print("-" * 50)
for rank, idx in enumerate(sorted_indices, 1):
    color = dominant_colors[idx]
    bright = brightness[idx]
    count = counts[np.where(unique == idx)[0][0]]
    print(f"{rank:<6} {color[0]:<6} {color[1]:<6} {color[2]:<6} {bright:<12} {count:<8}")

In [ ]:
# Teeth color is the brightest cluster
teeth_color_idx = sorted_indices[0]
teeth_color = dominant_colors[teeth_color_idx]

print(f"\n=== TEETH COLOR ===")
print(f"RGB: ({teeth_color[0]}, {teeth_color[1]}, {teeth_color[2]})")

# Convert RGB to HSV for better color description
teeth_bgr = cv2.cvtColor(np.uint8([[teeth_color[::-1]]]), cv2.COLOR_RGB2HSV)[0][0]
h, s, v = teeth_bgr

print(f"HSV: ({h}, {s}, {v})")

# Describe the color
if s < 30:  # Low saturation = grayish/white
    if v > 200:
        teeth_description = "Very Bright White"
    elif v > 150:
        teeth_description = "Bright White"
    elif v > 100:
        teeth_description = "Off-White / Cream"
    else:
        teeth_description = "Grayish White"
else:
    teeth_description = "Yellowish / Tinted"

print(f"Description: {teeth_description}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Row 1: Original data
axes[0, 0].imshow(smile_bbox)
axes[0, 0].set_title("Original Mouth Region", fontsize=11, fontweight='bold')
axes[0, 0].axis("off")

axes[0, 1].imshow(smile_mask_cropped, cmap="gray")
axes[0, 1].set_title("Mouth Mask", fontsize=11, fontweight='bold')
axes[0, 1].axis("off")

smile_extracted = np.zeros_like(smile_bbox)
smile_extracted[smile_mask_cropped] = smile_bbox[smile_mask_cropped]
axes[0, 2].imshow(smile_extracted)
axes[0, 2].set_title("Extracted Smile Part", fontsize=11, fontweight='bold')
axes[0, 2].axis("off")

# Row 2: Color clusters - reconstruct full image from sparse labels
cluster_img = np.zeros_like(smile_bbox)
for i in range(n_clusters):
    # Create mask for this cluster from the mouth pixels
    cluster_mask = (labels_km == i)
    # Place these pixels back into their original positions in smile_mask_cropped
    cluster_positions = smile_mask_cropped.copy()
    cluster_positions[smile_mask_cropped] = cluster_mask
    cluster_img[cluster_positions] = dominant_colors[i]

axes[1, 0].imshow(cluster_img)
axes[1, 0].set_title("K-means Clusters (n=5)", fontsize=11, fontweight='bold')
axes[1, 0].axis("off")

# Color palette of dominant colors (sorted by brightness)
color_palette = np.zeros((50, n_clusters * 50, 3), dtype=np.uint8)
for i, idx in enumerate(sorted_indices):
    color_palette[:, i*50:(i+1)*50] = dominant_colors[idx]

axes[1, 1].imshow(color_palette)
axes[1, 1].set_title("Dominant Colors (Brightest → Darkest)", fontsize=11, fontweight='bold')
axes[1, 1].set_xticks([])
axes[1, 1].set_yticks([])

# Teeth color highlight
teeth_swatch = np.zeros((100, 200, 3), dtype=np.uint8)
teeth_swatch[:] = teeth_color
axes[1, 2].imshow(teeth_swatch)
axes[1, 2].set_title(f"Teeth Color\n{teeth_description}\nRGB: {tuple(teeth_color)}",
                     fontsize=11, fontweight='bold')
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()


### 5. Smile Width Analysis
We measure mouth width by comparing it to the distance between the eyes
(interpupillary distance). Eye and mouth-corner positions are located with
MediaPipe FaceMesh (iris landmarks give precise pupil centers), since the
SegFormer mask only covers the mouth region and has no eye landmarks.

In [ ]:


import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh

# refine_landmarks=True is required to get iris landmarks (468, 473)
with mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
) as face_mesh:
    mesh_results = face_mesh.process(img)  # img: RGB array from section 4

if not mesh_results.multi_face_landmarks:
    raise RuntimeError("No face landmarks detected for smile width analysis.")

face_landmarks = mesh_results.multi_face_landmarks[0]
img_h, img_w = img.shape[:2]

def _to_px(idx):
    lm = face_landmarks.landmark[idx]
    return np.array([lm.x * img_w, lm.y * img_h])

# MediaPipe landmark ids
LEFT_IRIS_CENTER = 468
RIGHT_IRIS_CENTER = 473
MOUTH_LEFT_CORNER = 61
MOUTH_RIGHT_CORNER = 291

left_eye_pt = _to_px(LEFT_IRIS_CENTER)
right_eye_pt = _to_px(RIGHT_IRIS_CENTER)
mouth_left_pt = _to_px(MOUTH_LEFT_CORNER)
mouth_right_pt = _to_px(MOUTH_RIGHT_CORNER)

eye_spacing = float(np.linalg.norm(right_eye_pt - left_eye_pt))
mouth_width = float(np.linalg.norm(mouth_right_pt - mouth_left_pt))

mouth_to_eye_ratio = mouth_width / eye_spacing
IDEAL_RATIO = 1.0

if mouth_to_eye_ratio < 0.95:
    comparison = "narrower than"
elif mouth_to_eye_ratio > 1.05:
    comparison = "wider than"
else:
    comparison = "about equal to"

print("=== SMILE WIDTH ANALYSIS ===")
print(f"Interpupillary (eye) spacing : {eye_spacing:.1f}px")
print(f"Mouth width                  : {mouth_width:.1f}px")
print(f"Mouth-to-Interpupillary ratio: {mouth_to_eye_ratio:.2f} : 1.00  (ideal = 1.00 : 1.00)")
print(f"Your mouth width is {comparison} your eye spacing.")

# --- Draw measurement lines on the photo ---
annotated = img.copy()
cv2.line(annotated, tuple(left_eye_pt.astype(int)), tuple(right_eye_pt.astype(int)), (255, 255, 255), 2)
cv2.line(annotated, tuple(mouth_left_pt.astype(int)), tuple(mouth_right_pt.astype(int)), (255, 255, 255), 2)
for pt in (left_eye_pt, right_eye_pt, mouth_left_pt, mouth_right_pt):
    cv2.circle(annotated, tuple(pt.astype(int)), 3, (0, 200, 255), -1)

plt.figure(figsize=(5, 6))
plt.imshow(annotated)
plt.axis("off")
plt.title("Smile Width Analysis")
plt.show()

# --- Proportion comparison chart (Your Proportion vs Ideal Proportion) ---
fig, ax = plt.subplots(figsize=(6, 5))

x = np.arange(2)  # 0 = "Your Proportion", 1 = "Ideal Proportion"
bar_w = 0.35

ax.bar(x[0] - bar_w / 2, mouth_width, bar_w, color="#3d4f5c", label="Mouth Width")
ax.bar(x[0] + bar_w / 2, eye_spacing, bar_w, color="#c9d3db", label="Eye Spacing")
ax.bar(x[1] - bar_w / 2, eye_spacing, bar_w, color="#3d4f5c")
ax.bar(x[1] + bar_w / 2, eye_spacing, bar_w, color="#c9d3db")

ax.set_xticks(x)
ax.set_xticklabels(["Your Proportion", "Ideal Proportion"])
ax.set_ylabel("Pixels")
ax.set_title("Mouth Width vs Eye Spacing")
ax.legend()

for xi, (mv, ev) in enumerate([(mouth_width, eye_spacing), (eye_spacing, eye_spacing)]):
    ax.text(x[xi] - bar_w / 2, mv + max(mv, ev) * 0.02, f"{mv:.0f}", ha="center", fontsize=9)
    ax.text(x[xi] + bar_w / 2, ev + max(mv, ev) * 0.02, f"{ev:.0f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nVALUE  {mouth_to_eye_ratio:.2f} : 1.00   (yours)")
print(f"VALUE  {IDEAL_RATIO:.2f} : 1.00   (ideal)")



### 6. Front Teeth Alignment (Crowding / Spacing) Analysis
Looks at the visible edge of the front teeth and measures how much it
deviates from a smooth, evenly-aligned line — a simple proxy for crowding
or spacing along the biting edge.

In [ ]:

from scipy.signal import find_peaks
mouth_rgb = smile_bbox
mouth_region_mask = smile_mask_cropped

ys_m, xs_m = np.where(mouth_region_mask)
x_center = int(np.median(xs_m))

# use a small band of columns around the center for a stable measurement
band = 3
band_sel = (xs_m >= x_center - band) & (xs_m <= x_center + band)
band_ys = ys_m[band_sel]

y_top = int(band_ys.min())
y_bottom = int(band_ys.max())
mouth_height = y_bottom - y_top

print("=== MOUTH HEIGHT ===")
print(f"Mouth height: {mouth_height}px (measured at mouth center)")

# --- Draw a caliper-style vertical line at the mouth center ---
annotated_height = mouth_rgb.copy()
cap_w = 6

cv2.line(annotated_height, (x_center, y_top), (x_center, y_bottom), (255, 255, 255), 1)
cv2.line(annotated_height, (x_center - cap_w, y_top), (x_center + cap_w, y_top), (255, 255, 255), 1)
cv2.line(annotated_height, (x_center - cap_w, y_bottom), (x_center + cap_w, y_bottom), (255, 255, 255), 1)

plt.figure(figsize=(5, 3))
plt.imshow(annotated_height)
plt.axis("off")
plt.title(f"Mouth Height: {mouth_height}px")
plt.show()

In [ ]:

mouth_region_mask = smile_mask_cropped

ys_m, xs_m = np.where(mouth_region_mask)
x_center_local = int(np.median(xs_m))

# use a small band of columns around the center for a stable measurement
band = 3
band_sel = (xs_m >= x_center_local - band) & (xs_m <= x_center_local + band)
band_ys = ys_m[band_sel]

y_top_local = int(band_ys.min())
y_bottom_local = int(band_ys.max())
mouth_height = y_bottom_local - y_top_local

# translate crop-local coordinates (relative to smile_bbox) back to full-image coordinates
x_center = x_center_local + x1
y_top = y_top_local + y1
y_bottom = y_bottom_local + y1

print("=== MOUTH HEIGHT ===")
print(f"Mouth height: {mouth_height}px (measured at mouth center)")

# --- Draw a caliper-style vertical line at the mouth center, on the full photo ---
annotated_height = img.copy()
cap_w = 8

cv2.line(annotated_height, (x_center, y_top), (x_center, y_bottom), (255, 255, 255), 2)
cv2.line(annotated_height, (x_center - cap_w, y_top), (x_center + cap_w, y_top), (255, 255, 255), 2)
cv2.line(annotated_height, (x_center - cap_w, y_bottom), (x_center + cap_w, y_bottom), (255, 255, 255), 2)

plt.figure(figsize=(5, 6))
plt.imshow(annotated_height)
plt.axis("off")
plt.title(f"Mouth Height: {mouth_height}px")
plt.show()
